# 1. Import library

In [1]:
import os
import random
import datetime
import json
import pandas   as pd
import numpy    as np


from tqdm                       import tqdm
from preProcessingData          import PreProcessData
from utility.matpilotDraw       import MatPlot
from prettytable                import PrettyTable
from model.vaeConv1DCustom      import vaeConv1DCustom 
import umap
import plotly.express           as px
from sklearn.cluster            import KMeans
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


matPlot    = MatPlot() 
preData    = PreProcessData()

# 2. Preprocessing

In [6]:
rawData = preData.dataLoading()

Process:21695||2024-04-24 17:58:02,313-INFO [preProcessingData.py:dataLoading():175]:Loading data from files ...
Process:21695||2024-04-24 17:58:56,067-INFO [preProcessingData.py:dataLoading():181]:Finishing loading data ...


Index(['tiempo', 'Descripción Equipo', 'Modo de Falla', 'Comentario',
       'Especialidad', 'Motivo', 'Inicio Reportado', 'Fin Reportado',
       'corriente(A)', 'velocidad(rpm)', 'nivel_tolva_compensacion(%)',
       'Potencia(kW)', 'posicion_poste(%)', 'rendimiento(ton)',
       'presion_estanque_aire(bar)', 'Piñón_ext(°C)', 'Piñón_int(°C)',
       'Piñón_int(mm/s)', 'Piñón_ext(mm/s)', 'Eje_piñón_aceite(%)',
       'Lubricación_retorno(°C)', 'Lubricación_depósito(°C)',
       'Lubricación_entrada(°C)', 'Lubricación_principal(psi)',
       'Lubricación_principal(A)', 'Lubricación_respaldo(psi)',
       'Lubricación_respaldo(A)', 'Lubricación_post_filtros(psi)',
       'Lubricación_válvula_MPS(psi)', 'Lubricación_entrada_sup(psi)',
       'Lubricación_entrada_inf(psi)', 'Lubricación_entrada_sup(m3/h)',
       'Lubricación_entrada_inf(m3/h)', 'Lubricación_depósito(%)',
       'Motor_Fase1(A)', 'Motor_Fase2(A)', 'Motor_Fase3(A)', 'Motor_Fase1(V)',
       'Motor_Fase2(V)', 'Motor_Fase3(V

In [ ]:
preData.save2H5Format(rawData)

In [2]:
data = preData.loadDataFromH5Format()
print(data.info())

Process:21695||2024-04-24 17:56:38,319-INFO [preProcessingData.py:loadDataFromH5Format():201]:Loading data from H5 files ...
Process:21695||2024-04-24 17:56:38,435-INFO [preProcessingData.py:loadDataFromH5Format():204]:Finish loading data from H5 files ...


<class 'pandas.core.frame.DataFrame'>
Int64Index: 321435 entries, 0 to 321434
Data columns (total 32 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   corriente(A)                   321435 non-null  float64
 1   velocidad(rpm)                 321435 non-null  float64
 2   nivel_tolva_compensacion(%)    321435 non-null  float64
 3   Potencia(kW)                   321435 non-null  float64
 4   posicion_poste(%)              321435 non-null  float64
 5   rendimiento(ton)               321435 non-null  float64
 6   presion_estanque_aire(bar)     321435 non-null  float64
 7   Piñón_ext(°C)                  321435 non-null  float64
 8   Piñón_int(°C)                  321435 non-null  float64
 9   Piñón_int(mm/s)                321435 non-null  float64
 10  Piñón_ext(mm/s)                321435 non-null  float64
 11  Eje_piñón_aceite(%)            321435 non-null  float64
 12  Lubricación_retorno(°C)       

#### Remove redundant mode of failure (option)

In [ ]:
preData.rmRedundantMOF()

#### Manipulated and efficiency Variables

In [3]:
preData.splitInputandEfficiencyVar()

+-------------------------------+-----------------------------+
|         ManupulatedVar        |        EfficiencyVar        |
+-------------------------------+-----------------------------+
|          corriente(A)         | nivel_tolva_compensacion(%) |
|         velocidad(rpm)        |       rendimiento(ton)      |
|          Potencia(kW)         |        Piñón_ext(°C)        |
|       posicion_poste(%)       |        Piñón_int(°C)        |
|   presion_estanque_aire(bar)  |       Piñón_int(mm/s)       |
|      Eje_piñón_aceite(%)      |       Piñón_ext(mm/s)       |
|   Lubricación_principal(psi)  |   Lubricación_retorno(°C)   |
|    Lubricación_principal(A)   |   Lubricación_depósito(°C)  |
|   Lubricación_respaldo(psi)   |   Lubricación_entrada(°C)   |
|    Lubricación_respaldo(A)    |                             |
| Lubricación_post_filtros(psi) |                             |
|  Lubricación_válvula_MPS(psi) |                             |
|  Lubricación_entrada_sup(psi) |       

##### Draw variables

In [ ]:
# ============================
# Manipulated variables
# ----------------------------
for idxFeature in preData.manipulatedCr1:
    drawedData = preData.crusher1[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

In [ ]:
# ============================
# Efficiency variables
# ----------------------------
for idxFeature in preData.efficiencyCr1:
    drawedData = preData.crusher1[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

#### Sumarry MOF

In [4]:
preData.sumarryMOF()

+-------------------------------------------------------------------------------------------------------+-----------------------------------+-----------------------------------+
|                                              Elimination                                              |            Critical MOF           |              Used MOF             |
+-------------------------------------------------------------------------------------------------------+-----------------------------------+-----------------------------------+
|                                                Stand by                                               |         Válvula defectuosa        |         Válvula defectuosa        |
|                                        Stock de producto lleno                                        |       Mantención programada       |      Lubricación  deficiente      |
|                                            Cambio de Turno                                            |     

#### Data summarization

In [7]:
preData.sumarryDataFromFile()

All the mode of failure: ['Válvula defectuosa', 'Mantención programada', 'Falta de aceite hidráulico', 'Lubricación \xa0deficiente', 'Temperatura fuera de rango', 'Bomba hidráulica con fuga', 'Fuga de aceite', 'Falla en sistema hidráulico', 'Falla sistema de hidroset-poste', 'Mandibulas o revestimiento suelto']
+----+-----------------------------------+---------------------+---------------------+------------+-----------+
| No |                MOF                |     startingTime    |      endingTime     | staringRow | endingRow |
+----+-----------------------------------+---------------------+---------------------+------------+-----------+
| 1  |         Válvula defectuosa        | 2023-01-06 10:52:00 | 2023-01-06 15:13:00 |    4776    |    5037   |
| 2  |       Mantención programada       | 2023-01-19 00:54:00 | 2023-01-19 01:53:00 |   16978    |   17037   |
| 3  |       Mantención programada       | 2023-01-19 03:24:00 | 2023-01-19 23:12:00 |   17128    |   18316   |
| 4  |       Ma

## TRAIN VAE for reconstruction error

In [8]:
norData = preData.normalizeData(rawData[preData.crusher1Value])    # Normalize the data

Process:21695||2024-04-24 17:59:12,387-INFO [preProcessingData.py:normalizeData():426]:Normalizing the data ...


In [ ]:
# ============================
# Manipulated variables
# ----------------------------
for idxFeature in preData.manipulatedCr1:
    drawedData = norData[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

In [ ]:
# ============================
# Efficiency variables
# ----------------------------
for idxFeature in preData.efficiencyCr1:
    drawedData = norData[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

In [9]:
healthyStateData        = preData.getHealthyState(norData)

Process:21695||2024-04-24 17:59:16,532-INFO [preProcessingData.py:getHealthyState():343]:Getting critical trajectories ...


In [ ]:
# curData         = rawData[preData.crusher1Value]
# greaterThan500  = curData["corriente(A)"]== 0
# indices         = greaterThan500[greaterThan500].index.tolist()
# print(indices)
curData         = norData

healthyExcellPos = {"startingRow":  [18317, 28175,  43739,  64526,  87163,  107316, 115185, 133833, 225442, 280493, 310291], #277099 instead of 280493, but the current values is 0
                    "endingRow":    [27068, 42727,  63051,  86359,  98506,  115132, 132599, 157742, 249835, 283708, 321436],
                    "percent":      [0.5,   0.5,    0.5,    0.5,    0.5,    0.5,    0.5,    0.5,    0.3,    0.5,    0.7]}   # should subtract 2

healthyData =  pd.DataFrame(columns=preData.crusher1Value)
for idx in range(len(healthyExcellPos["startingRow"])):
    lenOfHealthyRow     = int((healthyExcellPos["endingRow"][idx] - healthyExcellPos["startingRow"][idx]) * healthyExcellPos["percent"][idx])

    startingPos         = healthyExcellPos["startingRow"][idx] -2
    endingPos           = healthyExcellPos["startingRow"][idx] - 2 + lenOfHealthyRow
    htEachPeriod        = curData[startingPos:endingPos]         
    healthyData         = pd.concat([healthyData,htEachPeriod],ignore_index=True)

In [ ]:
# ============================
# Manipulated variables
# ----------------------------
for idxFeature in preData.manipulatedCr1:
    drawedData = healthyStateData[idxFeature].to_numpy().astype(float)
    matPlot.draw1D(yCoor=drawedData,
                   title=idxFeature)

In [10]:
inputX                  = healthyStateData[preData.manipulatedCr1]
outputY                 = healthyStateData[preData.efficiencyCr1]

trainXY,valXY           = preData.splitTrainVal(inputX,outputY)                     # Split input and efficiency variables into train and validation dataset

tensorXY                = preData.splitAndPadSequences(trainXY)                # Split them into sequences

print(len(trainXY[0]))
print(tensorXY[1].shape)

63350
(63321, 30, 9)


In [ ]:
seqLength       = 30
overlap         = 29

sequences       = []
step_size       = seqLength - overlap
totalLength     = len(trainX)

for start in range(0, totalLength-overlap, step_size):
    end = start + seqLength
    # ----------------------
    # If the sequence would 
    # go beyond the data length, 
    # pad it
    # ----------------------
    if end > totalLength:                
        padding  = end - totalLength
        # sequence = torch.nn.functional.pad(dataX[:, start:], (0, padding), 'constant', 0)
    
    else:
        sequence = trainX[start:end].to_numpy()           # Get sentense
        
    sequences.append(sequence)


tensorX =   np.stack(sequences, axis=0)    
print(tensorX.shape)


### Model

In [11]:
with open('model/modelConfig.json', 'r') as configFile:
    config = json.load(configFile)

vaeConfig = config["vae"]
 
inputX                  = healthyStateData[preData.manipulatedCr1]
outputY                 = healthyStateData[preData.efficiencyCr1]

trainXY,valXY           = preData.splitTrainVal(inputX,outputY)                     # Split input and efficiency variables into train and validation dataset

seqTrainXY              = preData.splitAndPadSequences(trainXY)                     # Split them into sequences
seqValXY                = preData.splitAndPadSequences(valXY)                       # Split them into sequences

model                   = vaeConv1DCustom(vaeConfig)

print("The size of train data:  ",seqTrainXY[0].shape ,"(input)    ", seqTrainXY[1].shape,"(output)")
print("The size of val data:    ",seqValXY[0].shape ,"(input)    ", seqValXY[1].shape,"(output)")

model.summary()

The size of train data:   (63321, 30, 23) (input)     (63321, 30, 9) (output)
The size of val data:     (15809, 30, 23) (input)     (15809, 30, 9) (output)
Model: "encoder"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
encoder_input (InputLayer)      [(None, 30, 23)]     0                                            
__________________________________________________________________________________________________
enc_conv_0 (Conv1D)             (None, 15, 256)      17920       encoder_input[0][0]              
__________________________________________________________________________________________________
enc_conv_1 (Conv1D)             (None, 8, 128)       98432       enc_conv_0[0][0]                 
__________________________________________________________________________________________________
enc_conv_2 (Conv1D)             (No

### D. UMAP for entire dataset

In [ ]:
manipulatedCr1  = preData.crusher1[preData.manipulatedCr1]
maWTimeCr1      = manipulatedCr1[:]
print("The input variables shape",maWTimeCr1.shape)
umapReduction = umap.UMAP(  n_components    = 3,
                            n_neighbors     = 15,
                            min_dist        = 0.1)
maEmbedding       = umapReduction.fit_transform(maWTimeCr1)

In [ ]:
kmeans = KMeans(n_clusters=4)
kmeans.fit(maEmbedding)

labels = kmeans.labels_
centers = kmeans.cluster_centers_

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Color map for clusters
colors = ['r', 'g', 'b', 'y']

# Plot each data point
for i in range(len(maEmbedding)):
    ax.scatter(maEmbedding[i, 0], maEmbedding[i, 1], maEmbedding[i, 2], c=colors[labels[i]], marker='o')

# Plot the centroids
for i in range(4):
    ax.scatter(centers[i, 0], centers[i, 1], centers[i, 2], c=colors[i], marker='x', s=100, linewidths=3)

ax.set_title('Mode of condition')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_zlabel('Feature 3')
plt.show()

In [ ]:
efficiencyCr1   = preData.crusher1[preData.efficiencyCr1]
effWTimeCr1     = efficiencyCr1[:]
print("The efficieny variables shape",effWTimeCr1.shape)

umapReduction1 = umap.UMAP(  n_components=3,
                            n_neighbors=15,
                            min_dist=0.1)

effEmbedding      = umapReduction1.fit_transform(effWTimeCr1)

In [ ]:
kmeans1 = KMeans(n_clusters=4)
kmeans1.fit(effEmbedding)

labels = kmeans1.labels_
centers = kmeans1.cluster_centers_

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# Color map for clusters
colors = ['r', 'g', 'b', 'y']

# Plot each data point
for i in range(len(effEmbedding)):
    ax.scatter(effEmbedding[i, 0], effEmbedding[i, 1], effEmbedding[i, 2], c=colors[labels[i]], marker='o')

# Plot the centroids
for i in range(4):
    ax.scatter(centers[i, 0], centers[i, 1], centers[i, 2], c=colors[i], marker='x', s=100, linewidths=3)

ax.set_title('Mode of condition')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_zlabel('Feature 3')
plt.show()

#### c.1 Draw UMAP

In [ ]:
matPlot.draw3D(yCoor1 = maEmbedding[:,0],
               yCoor2 = maEmbedding[:,1],
               yCoor3 = maEmbedding[:,2],
               title  = "UMAP for Manipulated variables")

matPlot.draw3D(yCoor1 = effEmbedding[:,0],
               yCoor2 = effEmbedding[:,1],
               yCoor3 = effEmbedding[:,2],
               title  = "UMAP for efficient variables")

In [ ]:
maEmbedding_df  = pd.DataFrame(maEmbedding, columns=['UMAP_1', 'UMAP_2', 'UMAP_3'])
effEmbedding_df = pd.DataFrame(effEmbedding, columns=['UMAP_1', 'UMAP_2', 'UMAP_3'])
print(maEmbedding_df)
print(effEmbedding_df)
fig = px.scatter_3d(
    maEmbedding_df,
    x='UMAP_1',
    y='UMAP_2',
    z='UMAP_3',
    title='3D UMAP Embedding with Plotly'
)

fig.update_traces(marker=dict(size=3))
fig.show()

fig = px.scatter_3d(
    effEmbedding_df,
    x='UMAP_1',
    y='UMAP_2',
    z='UMAP_3',
    title='3D UMAP Embedding with Plotly'
)

fig.update_traces(marker=dict(size=3))
fig.show()